# Poisson Solve in 2D with pytential + volumential

This notebook demonstrates a coupled solve for

\[
-\Delta u = f \quad \text{in } \Omega, \qquad u=g \quad \text{on } \partial\Omega,
\]

with $\Omega$ the unit disk, using the three-stage workflow:

1. harmonic extension of boundary data $g$ via one SKIE (`pytential`)
2. volume potential for $f$ via FMM (`volumential`)
3. harmonic correction with boundary data $-V[f]|_{\partial\Omega}$ via a second SKIE


## Prerequisites

Run this notebook in an environment with:

- an OpenCL runtime (for example `pocl`)
- `volumential` test extras installed (for `meshmode` + `pytential`)

Example setup:

```bash
uv sync --active --extra test
```


In [ ]:
from __future__ import annotations

from functools import partial
import numpy as np

import pyopencl as cl
import pyopencl.array  # noqa: F401

from arraycontext import flatten, unflatten
from meshmode.dof_array import DOFArray
from meshmode.discretization import Discretization
from meshmode.discretization.poly_element import InterpolatoryQuadratureSimplexGroupFactory
from meshmode.mesh.generation import ellipse, make_curve_mesh

from boxtree import TreeBuilder
from boxtree.array_context import PyOpenCLArrayContext as BoxtreePyOpenCLArrayContext
from boxtree.bounding_box import make_bounding_box_dtype
from boxtree.tools import AXIS_NAMES
from boxtree.traversal import FMMTraversalBuilder

from pytools import single_valued
from pytools.obj_array import new_1d as obj_array_1d

from pytential.array_context import PyOpenCLArrayContext
from pytential.qbx import QBXLayerPotentialSource
from pytential.target import PointsTarget

import volumential.meshgen as mg
from volumential.expansion_wrangler_fpnd import (
    FPNDExpansionWrangler,
    FPNDTreeIndependentDataForWrangler,
)
from volumential.function_extension import compute_harmonic_extension
from volumential.nearfield_potential_table import DuffyBuildConfig
from volumential.table_manager import NearFieldInteractionTableManager
from volumential.volume_fmm import drive_volume_fmm

from sumpy.expansion import DefaultExpansionFactory
from sumpy.kernel import LaplaceKernel


In [ ]:
dim = 2
bbox_a, bbox_b = -1.25, 1.25
q_order = 4
n_levels = 3
fmm_order = 12

table_filename = "nft_poisson2d_demo.sqlite"
table_root_extent = bbox_b - bbox_a

nelements_bdry = 64
bdry_order = 4
qbx_order = 3


def u_exact(x, y):
    return x**3 + y**2 - 0.5 * x * y + 0.25 * x**2 * y


def rhs_f(x, y):
    # -Delta(u_exact)
    return -(6.0 * x + 0.5 * y + 2.0)


In [ ]:
ctx = cl.create_some_context()
queue = cl.CommandQueue(ctx)
actx = PyOpenCLArrayContext(queue)

bdry_mesh = make_curve_mesh(
    partial(ellipse, 1.0),
    np.linspace(0.0, 1.0, nelements_bdry + 1),
    bdry_order,
)
density_discr = Discretization(
    actx,
    bdry_mesh,
    InterpolatoryQuadratureSimplexGroupFactory(bdry_order),
)
qbx = QBXLayerPotentialSource(
    density_discr,
    fine_order=4 * bdry_order,
    qbx_order=qbx_order,
    fmm_order=False,
)

bdry_nodes = actx.thaw(density_discr.nodes())
g_bdry = u_exact(bdry_nodes[0], bdry_nodes[1])


In [ ]:
volume_mesh = mg.MeshGen2D(
    degree=q_order,
    nlevels=n_levels,
    a=bbox_a,
    b=bbox_b,
    queue=queue,
)

source_points_host = np.ascontiguousarray(volume_mesh.get_q_points())
source_weights_host = np.ascontiguousarray(volume_mesh.get_q_weights())
eval_points_host = np.ascontiguousarray(volume_mesh.get_cell_centers())

target_eval = PointsTarget(actx.freeze(actx.from_numpy(eval_points_host.T)))

h_eval, dbg_h = compute_harmonic_extension(
    queue,
    target_eval,
    qbx,
    density_discr,
    g_bdry,
    loc_sign=-1,
    target_association_tolerance=0.05,
    gmres_tolerance=1e-10,
    actx=actx,
)
h_eval_host = actx.to_numpy(h_eval)

print("SKIE #1 (harmonic extension) state:", dbg_h["gmres_result"].state)
print("n source points:", source_points_host.shape[0], "n eval points:", eval_points_host.shape[0])


In [ ]:
flat_bdry_nodes = flatten(bdry_nodes, actx, leaf_class=DOFArray)
bdry_x_host = actx.to_numpy(flat_bdry_nodes[0])
bdry_y_host = actx.to_numpy(flat_bdry_nodes[1])
bdry_points_host = np.ascontiguousarray(np.vstack([bdry_x_host, bdry_y_host]))

all_targets_host = np.ascontiguousarray(
    np.hstack([eval_points_host.T, bdry_points_host])
)

source_points = obj_array_1d(
    [
        cl.array.to_device(queue, np.ascontiguousarray(source_points_host[:, iaxis]))
        for iaxis in range(dim)
    ]
)
target_points = obj_array_1d(
    [
        cl.array.to_device(queue, np.ascontiguousarray(all_targets_host[iaxis]))
        for iaxis in range(dim)
    ]
)

source_vals_host = rhs_f(source_points_host[:, 0], source_points_host[:, 1])
source_vals = cl.array.to_device(queue, np.ascontiguousarray(source_vals_host))
source_weights = cl.array.to_device(queue, source_weights_host)


In [ ]:
bt_actx = BoxtreePyOpenCLArrayContext(queue)
tb = TreeBuilder(bt_actx)

coord_dtype = single_valued(coord.dtype for coord in source_points)
bbox_type, _ = make_bounding_box_dtype(ctx.devices[0], dim, coord_dtype)
bbox = np.empty(1, bbox_type)
for ax in AXIS_NAMES[:dim]:
    bbox["min_" + ax] = bbox_a
    bbox["max_" + ax] = bbox_b

tree, _ = tb(
    bt_actx,
    particles=source_points,
    targets=target_points,
    bbox=bbox,
    max_particles_in_box=q_order**dim * (2**dim) - 1,
    kind="adaptive-level-restricted",
)

trav_builder = FMMTraversalBuilder(bt_actx)
trav, _ = trav_builder(bt_actx, tree)

print("tree levels:", tree.nlevels, "boxes:", tree.nboxes, "targets:", tree.ntargets)


In [ ]:
tm = NearFieldInteractionTableManager(
    table_filename,
    root_extent=table_root_extent,
    queue=queue,
)
build_config = DuffyBuildConfig(
    radial_rule="tanh-sinh-fast",
    regular_quad_order=16,
    radial_quad_order=61,
)

nftable, _ = tm.get_table(
    dim,
    "Laplace",
    q_order,
    force_recompute=False,
    queue=queue,
    build_config=build_config,
)

knl = LaplaceKernel(dim)
expn_factory = DefaultExpansionFactory()
local_expn_class = expn_factory.get_local_expansion_class(knl)
mpole_expn_class = expn_factory.get_multipole_expansion_class(knl)

tree_indep = FPNDTreeIndependentDataForWrangler(
    ctx,
    partial(mpole_expn_class, knl),
    partial(local_expn_class, knl),
    [knl],
    exclude_self=False,
)

wrangler = FPNDExpansionWrangler(
    tree_indep=tree_indep,
    traversal=trav,
    near_field_table=nftable,
    dtype=np.float64,
    fmm_level_to_order=lambda kernel, kernel_args, tree, lev: fmm_order,
    quad_order=q_order,
    queue=queue,
)


In [ ]:
(vf_all,) = drive_volume_fmm(
    trav,
    wrangler,
    source_vals * source_weights,
    source_vals,
)

vf_all_host = vf_all.get() if hasattr(vf_all, "get") else np.asarray(vf_all)
n_eval = eval_points_host.shape[0]

vf_eval_host = vf_all_host[:n_eval]
vf_bdry_host = vf_all_host[n_eval:]

print("computed volume potential on", n_eval, "volume targets and", vf_bdry_host.size, "boundary targets")


In [ ]:
corr_bdry_flat = actx.from_numpy(np.ascontiguousarray(-vf_bdry_host))
corr_bdry = unflatten(bdry_nodes[0], corr_bdry_flat, actx)

corr_eval, dbg_corr = compute_harmonic_extension(
    queue,
    target_eval,
    qbx,
    density_discr,
    corr_bdry,
    loc_sign=-1,
    target_association_tolerance=0.05,
    gmres_tolerance=1e-10,
    actx=actx,
)
corr_eval_host = actx.to_numpy(corr_eval)

u_num_host = h_eval_host + vf_eval_host + corr_eval_host
u_ref_host = u_exact(eval_points_host[:, 0], eval_points_host[:, 1])

inside_mask = eval_points_host[:, 0] ** 2 + eval_points_host[:, 1] ** 2 <= 1.0
abs_err = np.abs(u_num_host - u_ref_host)
rel_l2 = np.linalg.norm(abs_err[inside_mask]) / np.linalg.norm(u_ref_host[inside_mask])

print("SKIE #2 (boundary correction) state:", dbg_corr["gmres_result"].state)
print("inside-domain max abs error:", abs_err[inside_mask].max())
print("inside-domain rel L2 error:", rel_l2)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

sc0 = ax[0].scatter(
    eval_points_host[:, 0],
    eval_points_host[:, 1],
    c=u_num_host,
    s=20,
    cmap="viridis",
)
ax[0].set_title("u_num on volume targets")
ax[0].set_aspect("equal")
fig.colorbar(sc0, ax=ax[0], shrink=0.8)

sc1 = ax[1].scatter(
    eval_points_host[:, 0],
    eval_points_host[:, 1],
    c=np.where(inside_mask, abs_err, np.nan),
    s=20,
    cmap="magma",
)
ax[1].set_title("|u_num - u_exact| (inside unit disk)")
ax[1].set_aspect("equal")
fig.colorbar(sc1, ax=ax[1], shrink=0.8)

plt.show()
